In [7]:
import pandas as pd

df_clean = pd.read_csv(
    "../data/processed/cleaned_messages.csv"
)


In [8]:
import sklearn.model_selection
from sklearn.model_selection import train_test_split 

X = df_clean["message"]
y = df_clean["label"]

X_train, X_test, y_train, y_test = train_test_split(
                                                     X, y, test_size = 0.2, 
                                                     random_state = 42, stratify = y
                                                    )


In [9]:
print(X_test.shape)
print(X_train.shape)

(1034,)
(4135,)


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [11]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(4135, 7680)
(1034, 7680)


In [12]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

Checking 10 messages's predicted y(labels)

In [13]:
print(y_pred[:10])

['ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [14]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.9526112185686654

In [15]:
print(y_test[:10].tolist())
print(y_pred[:10].tolist())

['ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham']
['ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham']


In [16]:
import pandas as pd

print(pd.Series(y_pred).value_counts())

ham     952
spam     82
Name: count, dtype: int64


In [17]:
print(type(y_test))
print(type(y_pred))
print(accuracy_score(y_test, y_pred))

<class 'pandas.Series'>
<class 'numpy.ndarray'>
0.9526112185686654


In [18]:
print((y_test == y_pred).sum())
print(len(y_test))

985
1034


In [19]:
print(y_test[:20].tolist())
print(y_train[:10])

['ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham']
331     ham
383     ham
573     ham
1346    ham
1070    ham
4664    ham
4356    ham
4895    ham
1636    ham
3121    ham
Name: label, dtype: str


In [20]:
print((y_test == y_pred).all())
print(y_test is y_pred)

False
False


**Logistic Regession**

In [21]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

In [22]:
accuracy_score(y_test, y_pred_lr)

0.9642166344294004

In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       903
        spam       0.96      0.75      0.84       131

    accuracy                           0.96      1034
   macro avg       0.96      0.87      0.91      1034
weighted avg       0.96      0.96      0.96      1034



In [24]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       903
        spam       1.00      0.63      0.77       131

    accuracy                           0.95      1034
   macro avg       0.97      0.81      0.87      1034
weighted avg       0.96      0.95      0.95      1034



In [25]:
print((y_pred == y_pred_lr).all())

False


In [26]:
from sklearn.svm import LinearSVC

svc_model = LinearSVC()
svc_model.fit(X_train_tfidf, y_train)

y_pred_svc = svc_model.predict(X_test_tfidf)

In [27]:
accuracy_score(y_test, y_pred_svc)

0.9816247582205029

In [28]:
print(classification_report(y_test, y_pred_svc))

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       903
        spam       0.97      0.89      0.92       131

    accuracy                           0.98      1034
   macro avg       0.98      0.94      0.96      1034
weighted avg       0.98      0.98      0.98      1034



In [29]:
new_message = ["Congratulations! You have won a free prize. Click here now!"]

In [30]:
new_message_tfidf = vectorizer.transform(new_message)

In [31]:
prediction = lr_model.predict(new_message_tfidf)

if prediction[0] == "spam":
    print("Spam message detected")
else:
    print("Ham message (Not Spam)")

Spam message detected


In [32]:
new_message_1 = ["Hey, are we meeting today?"]
new_message_1_tfidf = vectorizer.transform(new_message_1)
prediction_1 = lr_model.predict(new_message_1_tfidf)

if prediction_1[0] == "spam":
    print("Spam message detected")
else:
    print("Ham message (Not Spam)")

Ham message (Not Spam)


# predict() का output array में आता है, भले ही हमने सिर्फ एक message दिया हो, इसलिए output ['ham'] है।
# prediction[0] में [0] का मतलब है array का पहला result, क्योंकि Python में indexing 0 से शुरू होती है।

In [33]:
lr_model.predict_proba(new_message_tfidf)

array([[0.34350442, 0.65649558]])

In [34]:
spam_probability = lr_model.predict_proba(new_message_tfidf)[0][1]
print(f"Spam Probability: {spam_probability:.2%}")

Spam Probability: 65.65%


In [35]:
lr_model.classes_

array(['ham', 'spam'], dtype=object)

In [36]:
probabilities = lr_model.predict_proba(new_message_tfidf)
print(f"Ham Probability: {probabilities[0][0]:.2%}")
print(f"Spam Probability: {probabilities[0][1]:.2%}")

Ham Probability: 34.35%
Spam Probability: 65.65%


In [37]:
if spam_probability >= 0.70:
    print('Spam')
else:
    print('Review Needed')

Review Needed


In [38]:
def detect_message(message):
    message_tfidf = vectorizer.transform([message])
    probabilities = lr_model.predict_proba(message_tfidf)
    spam_probability = probabilities[0][1]
    ham_probability = probabilities[0][0]

    if spam_probability >= 0.70:
        return "Spam", f"{spam_probability:.2%}"
    elif ham_probability >= 0.70:
        return "Ham", f"{ham_probability:.2%}"
    else:
        return "Review Needed"

In [39]:
detect_message("Hey, are we eating today?")
detect_message('Wow! You should download this app to have free coupon.')
detect_message('I will call you when I reach school.')

('Ham', '89.05%')

In [40]:
errors = y_test != y_pred_lr


In [41]:
error_table = pd.DataFrame({
    "Message": X_test[errors],
    "Actual": y_test[errors],
    "Predicted": y_pred_lr[errors]
})

error_table

,Message,Actual,Predicted
5137,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham
521,You will recieve your tone within the next 24h...,spam,ham
2697,Send a logo 2 ur lover - 2 names joined by a h...,spam,ham
484,Congrats! 1 year special cinema pass for 2 is ...,spam,ham
1611,Hi if ur lookin 4 saucy daytime fun wiv busty ...,spam,ham
4752,TheMob>Hit the link to get a premium Pink Pant...,spam,ham
1405,As a registered optin subscriber ur draw 4 £10...,spam,ham
5139,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...,spam,ham
4365,Customer service announcement. We recently tri...,spam,ham
1645,"Free msg. Sorry, a service you ordered from 81...",spam,ham


In [42]:
error_table.shape[0]

error_table['Actual'].value_counts()

error_table['Predicted'].value_counts()

error_table.head(10)



,Message,Actual,Predicted
5137,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham
521,You will recieve your tone within the next 24h...,spam,ham
2697,Send a logo 2 ur lover - 2 names joined by a h...,spam,ham
484,Congrats! 1 year special cinema pass for 2 is ...,spam,ham
1611,Hi if ur lookin 4 saucy daytime fun wiv busty ...,spam,ham
4752,TheMob>Hit the link to get a premium Pink Pant...,spam,ham
1405,As a registered optin subscriber ur draw 4 £10...,spam,ham
5139,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...,spam,ham
4365,Customer service announcement. We recently tri...,spam,ham
1645,"Free msg. Sorry, a service you ordered from 81...",spam,ham


In [43]:
(error_table['Actual'] == "spam" ).sum() /len(error_table) * 100

np.float64(89.1891891891892)

In [44]:
error_table[error_table['Actual'] == "spam" ][["Message", "Actual", 'Predicted']].head(5)

,Message,Actual,Predicted
5137,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham
521,You will recieve your tone within the next 24h...,spam,ham
2697,Send a logo 2 ur lover - 2 names joined by a h...,spam,ham
484,Congrats! 1 year special cinema pass for 2 is ...,spam,ham
1611,Hi if ur lookin 4 saucy daytime fun wiv busty ...,spam,ham


In [45]:
missed_spam = error_table[
    (error_table['Actual'] == 'spam') & 
    (error_table['Predicted'] == 'ham')
]
missed_spam['Message'].head(10)

5137    Want explicit SEX in 30 secs? Ring 02073162414...
521     You will recieve your tone within the next 24h...
2697    Send a logo 2 ur lover - 2 names joined by a h...
484     Congrats! 1 year special cinema pass for 2 is ...
1611    Hi if ur lookin 4 saucy daytime fun wiv busty ...
4752    TheMob>Hit the link to get a premium Pink Pant...
1405    As a registered optin subscriber ur draw 4 £10...
5139    ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
4365    Customer service announcement. We recently tri...
1645    Free msg. Sorry, a service you ordered from 81...
Name: Message, dtype: str

# New Vectorizer

In [46]:
vectorizer_v2 = TfidfVectorizer(
    analyzer = 'char',
    ngram_range = (3, 5),
    min_df = 2
)

In [47]:
X_train_v2 = vectorizer_v2.fit_transform(X_train)
X_test_V2 = vectorizer_v2.transform(X_test)

In [48]:
lr_model_v2 = LogisticRegression(max_iter = 1000)
lr_model_v2.fit(X_train_v2, y_train)

y_pred_lr_v2 = lr_model_v2.predict(X_test_V2)

In [49]:
print(classification_report(y_test, y_pred_lr_v2))

              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       903
        spam       1.00      0.76      0.87       131

    accuracy                           0.97      1034
   macro avg       0.98      0.88      0.92      1034
weighted avg       0.97      0.97      0.97      1034



In [50]:
print('OLD MODEL')
print(classification_report(y_test, y_pred_lr))

print('NEW MODEL')
print(classification_report(y_test, y_pred_lr_v2))

OLD MODEL
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       903
        spam       0.96      0.75      0.84       131

    accuracy                           0.96      1034
   macro avg       0.96      0.87      0.91      1034
weighted avg       0.96      0.96      0.96      1034

NEW MODEL
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       903
        spam       1.00      0.76      0.87       131

    accuracy                           0.97      1034
   macro avg       0.98      0.88      0.92      1034
weighted avg       0.97      0.97      0.97      1034



In [51]:
import joblib

joblib.dump(lr_model_v2, "osmd_model.pkl")
joblib.dump(vectorizer_v2, "osmd_vectorizer.pkl")

['osmd_vectorizer.pkl']

In [52]:
loaded_model = joblib.load("osmd_model.pkl")
loaded_vectorizer = joblib.load("osmd_vectorizer.pkl")

In [53]:
message = ["Congratulations! You have won a free prize. Click here now!"]

message_tfidf = loaded_vectorizer.transform(message)

prediction = loaded_model.predict(message_tfidf)

print("Prediction:", prediction[0])

Prediction: spam


In [54]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_lr_v2, labels = ["ham", "spam"])
print(cm)

[[903   0]
 [ 31 100]]


In [55]:
cm_old = confusion_matrix(y_test, y_pred_lr, labels = ["ham", "spam"])
print(cm_old)

[[899   4]
 [ 33  98]]


In [56]:
cm_nb = confusion_matrix(y_test, y_pred, labels = ["ham", "spam"])
print(cm_nb)

[[903   0]
 [ 49  82]]


In [57]:
cm_svc = confusion_matrix(y_test, y_pred_svc, labels = ["ham", "spam"])
print(cm_svc)

[[899   4]
 [ 15 116]]


In [58]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("SVC")
print("\nAccuracy:", accuracy_score(y_test, y_pred_svc))
print("Precision:", precision_score(y_test, y_pred_svc, pos_label = "spam"))
print("Recall:", recall_score(y_test, y_pred_svc, pos_label = "spam"))
print("F1-Score:", f1_score(y_test, y_pred_svc, pos_label = "spam"))

print("\nNEW LOGISTIC REGRESSION")
print("\nAccuracy:", accuracy_score(y_test, y_pred_lr_v2))
print("Precision:", precision_score(y_test, y_pred_lr_v2, pos_label = "spam"))
print("Recall:", recall_score(y_test, y_pred_lr_v2, pos_label = "spam"))
print("F1-Score:", f1_score(y_test, y_pred_lr_v2, pos_label = "spam"))


SVC

Accuracy: 0.9816247582205029
Precision: 0.9666666666666667
Recall: 0.8854961832061069
F1-Score: 0.9243027888446215

NEW LOGISTIC REGRESSION

Accuracy: 0.9700193423597679
Precision: 1.0
Recall: 0.7633587786259542
F1-Score: 0.8658008658008658


In [59]:
print("SVC")
print(classification_report(y_test, y_pred_svc))

SVC
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       903
        spam       0.97      0.89      0.92       131

    accuracy                           0.98      1034
   macro avg       0.98      0.94      0.96      1034
weighted avg       0.98      0.98      0.98      1034



# SVC New Vectorizer

In [60]:
vectorizer_svc2 = TfidfVectorizer(
    analyzer = 'char',
    ngram_range = (3, 5),
    min_df = 2)

In [61]:
X_train_svc2 = vectorizer_svc2.fit_transform(X_train)
X_test_svc2 = vectorizer_svc2.transform(X_test)

In [62]:
svc_model_2 = LinearSVC(max_iter = 1000 )
svc_model_2.fit(X_train_svc2, y_train)

y_svc_pred_2 = svc_model_2.predict(X_test_svc2)

In [63]:
print("NEW SVC")
print(classification_report(y_test, y_svc_pred_2))

NEW SVC
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       903
        spam       0.99      0.90      0.94       131

    accuracy                           0.99      1034
   macro avg       0.99      0.95      0.97      1034
weighted avg       0.99      0.99      0.99      1034



In [64]:
cm_svc = confusion_matrix(y_test, y_pred_svc, labels = ["ham", "spam"])
print(cm_svc)

[[899   4]
 [ 15 116]]


In [65]:
cm_svc_new = confusion_matrix(y_test, y_svc_pred_2, labels = ['ham', 'spam'])
print(cm_svc_new)

[[902   1]
 [ 13 118]]


In [66]:
print("SVC")
print("\nAccuracy:", accuracy_score(y_test, y_pred_svc))
print("Precision:", precision_score(y_test, y_pred_svc, pos_label = "spam"))
print("Recall:", recall_score(y_test, y_pred_svc, pos_label = "spam"))
print("F1-Score:", f1_score(y_test, y_pred_svc, pos_label = "spam"))

print("\n NEW SVC")
print("\nAccuracy:", accuracy_score(y_test, y_svc_pred_2))
print("Precision:", precision_score(y_test, y_svc_pred_2, pos_label = "spam"))
print("Recall:", recall_score(y_test, y_svc_pred_2, pos_label = "spam"))
print("F1-Score:", f1_score(y_test, y_svc_pred_2, pos_label = "spam"))

SVC

Accuracy: 0.9816247582205029
Precision: 0.9666666666666667
Recall: 0.8854961832061069
F1-Score: 0.9243027888446215

 NEW SVC

Accuracy: 0.9864603481624759
Precision: 0.9915966386554622
Recall: 0.9007633587786259
F1-Score: 0.944


# Error Analysis of New Char SVC

In [67]:
errors_svc_2 = y_test != y_svc_pred_2
errors_svc_2

4940    False
4908    False
2358    False
1267    False
4738    False
        ...  
1095    False
2912    False
796     False
1511    False
1203    False
Name: label, Length: 1034, dtype: bool

In [68]:
errors_svc_2.sum()

np.int64(14)

In [69]:
error_table_svc =  pd.DataFrame({
    "Messages": X_test,
    "Actual_labels": y_test,
    "New_svc_predictions": y_svc_pred_2})


In [70]:
error_table_svc =error_table_svc[errors_svc_2]

In [71]:
error_table_svc.shape

(14, 3)

In [72]:
error_table_svc

,Messages,Actual_labels,New_svc_predictions
4752,TheMob>Hit the link to get a premium Pink Pant...,spam,ham
5139,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...,spam,ham
3949,Missed call alert. These numbers called but le...,spam,ham
2303,Babe: U want me dont u baby! Im nasty and have...,spam,ham
4499,Check Out Choose Your Babe Videos @ sms.shsex....,spam,ham
2180,88066 FROM 88066 LOST 3POUND HELP,spam,ham
3890,In The Simpsons Movie released in July 2007 na...,spam,ham
3332,Xmas & New Years Eve tickets are now on sale f...,spam,ham
5062,"Latest News! Police station toilet stolen, cop...",spam,ham
4185,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham


In [73]:
missed_spam_svc = error_table_svc[
    (error_table_svc["Actual_labels"] == "spam") &
    (error_table_svc["New_svc_predictions"] == "ham")
]

In [74]:
missed_spam_svc.shape

(13, 3)

In [75]:
print(missed_spam_svc.to_string(index = False))

                                                                                                                                                      Messages Actual_labels New_svc_predictions
         TheMob>Hit the link to get a premium Pink Panther game, the new no. 1 from Sugababes, a crazy Zebra animation or a badass Hoody wallpaper-all 4 FREE!          spam                 ham
ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE MINS. INDIA CUST SERVs SED YES. L8ER GOT MEGA BILL. 3 DONT GIV A SHIT. BAILIFF DUE IN DAYS. I O £250 3 WANT £800          spam                 ham
                                                                                      Missed call alert. These numbers called but left no message. 07008009200          spam                 ham
      Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about we go slo n hard! Txt XXX SLO(4msgs)          spam                 ham
                                   

# to_string() = टेबल को text में दिखाना; index=False = बाईं तरफ के index numbers छिपाना।
# Here missed_spam_svc is False Negative

In [76]:
missed_ham_svc = error_table_svc[
    (error_table_svc["Actual_labels"] == "ham") &
    (error_table_svc["New_svc_predictions"] == "spam")
]

In [77]:
missed_ham_svc.shape

(1, 3)

In [78]:
print(missed_ham_svc.to_string(index = False))

                                                                                                                    Messages Actual_labels New_svc_predictions
I (Career Tel) have added u as a contact on INDYAROCKS.COM to send FREE SMS. To remove from phonebook - sms NO to  &lt;#&gt;           ham                spam


# Here missed_ham_svc is False Positive

In [79]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score, StratifiedKFold


# यह तय करता है कि data को पाँच folds में कैसे बाँटना है।
# हमारे dataset में ham messages ज़्यादा हैं और spam कम। Stratified K-Fold हर fold में इन दोनों classes का proportion लगभग समान रखने की कोशिश करता है।
# उदाहरण के लिए, अगर training data में लगभग 87% ham और 13% spam हैं, तो हर fold में भी अनुपात लगभग यही रहेगा।
# इससे कोई fold गलती से बहुत ज़्यादा ham या बहुत कम spam वाला नहीं बनता।

In [80]:
osmd_pipeline = make_pipeline(
    TfidfVectorizer(
        analyzer = "char",
        ngram_range = (3, 5),
        min_df = 2
    ),
    LinearSVC()
)

In [81]:
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

# shuffle = True → data का क्रम मिलाओ।
# random_state = 42 → वही random क्रम दोबारा पाने में मदद करो।

In [82]:
from sklearn.metrics import make_scorer, f1_score

# ham = 0 नहीं, बल्कि अभी actual label "ham" है।
# spam = 1 नहीं, बल्कि actual label "spam" है।
# यानी y_train में labels "ham" और "spam" strings के रूप में हैं, numeric 0 और 1 के रूप में नहीं।
# जब scoring="f1" लिखते हो, तो Scikit-learn डिफ़ॉल्ट रूप से 1 को positive label मानता है। लेकिन तुम्हारे dataset में 1 नाम का label है ही नहीं, इसलिए error आया।

In [83]:
spam_f1_scorer = make_scorer(f1_score, pos_label = "spam")

# make_scorer(...) — Scikit-learn को बताता है कि कौन-सा scoring function इस्तेमाल करना है।
# f1_score — बताता है कि F1 Score निकालना है।
# pos_label="spam" — बताता है कि Spam class को positive मानकर F1 Score निकालना है।
# spam_f1_scorer — उस तैयार scorer को दिया गया variable name है। इसे तुम चाहो तो दूसरा नाम भी दे सकते हो।

In [84]:
cv_scores = cross_val_score(
    osmd_pipeline,
    X_train, y_train, cv = cv,
    scoring = spam_f1_scorer,
    n_jobs = -1
)

# osmd_pipeline — TF-IDF + LinearSVC।
# X_train, y_train — केवल training data।
# cv=cv — हमारे तय किए हुए पाँच folds।
# scoring="f1" — हर round में Spam class का F1 score।
# n_jobs=-1 — उपलब्ध CPU cores का उपयोग करके calculation तेज़ करने की कोशिश।

In [85]:
cv_scores

array([0.94949495, 0.93877551, 0.94949495, 0.95566502, 0.94527363])

# इसका मतलब Cross-Validation के दौरान अभी भी कोई error हुआ है। nan का अर्थ है कि score की valid numerical value नहीं मिली।

In [86]:
cv_scores = cross_val_score(
    osmd_pipeline,
    X_train, y_train, cv = cv,
    scoring = spam_f1_scorer,
    n_jobs = -1,
    error_score = "raise"
)

In [87]:
print(cv_scores)

[0.94949495 0.93877551 0.94949495 0.95566502 0.94527363]


In [88]:
print(cv_scores.mean())

0.9477408131330638


In [89]:
print(cv_scores.std())

0.0055751368906292375


# standard deviation मिलेगा—यानी पाँचों F1 scores में कितना उतार-चढ़ाव है। Output भेजो।

# Hyperparameter Tuning

In [90]:
from  sklearn.model_selection import GridSearchCV

In [91]:
param_grid = {
    "linearsvc__C": [0.1, 1, 10]
}

In [92]:
grid_search = GridSearchCV(
    estimator = osmd_pipeline,
    param_grid = param_grid,
    scoring = spam_f1_scorer,
    cv = cv,
    n_jobs = -1,
    error_score = "raise"
)

In [93]:
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...LinearSVC())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'linearsvc__C': [0.1, 1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",make_scorer(f...os_label=spam)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callabl

In [94]:
grid_search.best_params_

{'linearsvc__C': 10}

In [95]:
grid_search.best_score_

np.float64(0.9562205739070263)

In [96]:
best_osmd_model = grid_search.best_estimator_

y_tuned_pred = best_osmd_model.predict(X_test)

# Confusion Matrix

In [97]:
from sklearn.metrics import confusion_matrix

cm_tuned = confusion_matrix(
    y_test, 
    y_tuned_pred,
    labels = ['ham','spam']
)

In [98]:
print(cm_tuned)

[[902   1]
 [ 11 120]]


In [99]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_tuned_pred,
    labels=["ham", "spam"],
    zero_division=0
))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       903
        spam       0.99      0.92      0.95       131

    accuracy                           0.99      1034
   macro avg       0.99      0.96      0.97      1034
weighted avg       0.99      0.99      0.99      1034



# Comparsion c = 1 and c = 10

In [100]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Spam Precision", "Spam Recall", "Spam F1"],
    "Old SVC (C=1)": [
        accuracy_score(y_test, y_svc_pred_2),
        precision_score(y_test, y_svc_pred_2, pos_label="spam"),
        recall_score(y_test, y_svc_pred_2, pos_label="spam"),
        f1_score(y_test, y_svc_pred_2, pos_label="spam")
    ],
    "Tuned SVC (C=10)": [
        accuracy_score(y_test, y_tuned_pred),
        precision_score(y_test, y_tuned_pred, pos_label="spam"),
        recall_score(y_test, y_tuned_pred, pos_label="spam"),
        f1_score(y_test, y_tuned_pred, pos_label="spam")
    ]
})

comparison

,Metric,Old SVC (C=1),Tuned SVC (C=10)
0,Accuracy,0.986460,0.988395
1,Spam Precision,0.991597,0.991736
2,Spam Recall,0.900763,0.916031
3,Spam F1,0.944000,0.952381


In [101]:
import joblib

joblib.dump(best_osmd_model, "osmd_tuned_pipeline.pkl")

['osmd_tuned_pipeline.pkl']

In [102]:
import os
print(os.path.exists("osmd_tuned_pipeline.pkl"))

True
